# LeetCode 546 — Remove Boxes bằng Python

- Link: https://leetcode.com/problems/remove-boxes/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên đoạn (Interval DP), Đệ quy có nhớ (Memoization), DP 3 chiều

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def removeBoxes(self, boxes: List[int]) -> int:

```

## 1. Giới thiệu & mục tiêu học

Đây là một trong những bài **DP trên đoạn** nổi tiếng là "khó nghĩ" nhất của LeetCode. Cái khó không nằm ở code (code rất ngắn) mà ở chỗ **trạng thái DP hiển nhiên `dp[l][r]` là không đủ** — ta phải thêm một chiều thứ ba để "nhớ" thông tin bên ngoài đoạn.

Sau notebook này bạn sẽ:

1. Hiểu vì sao bài toán **không tách được** thành hai nửa độc lập như các bài DP đoạn thông thường.
2. Biết cách **thêm thông tin vào trạng thái** (`k` hộp cùng màu đang "dính" bên trái) để bài toán con trở nên tự đủ — kỹ thuật dùng lại được ở nhiều bài khó khác.
3. Viết được 4 lời giải: vét cạn có nhớ trên tuple, đệ quy có nhớ `dp(l, r, k)`, DP bảng 3 chiều (bottom-up), và DP trên **nhóm đã nén** (run-length).
4. Biết trace một hàm đệ quy có nhớ bằng cách in cây lời gọi.
5. Kiểm chứng lời giải bằng brute force trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có một dãy hộp `boxes`, mỗi hộp có một màu (số nguyên dương). Ở mỗi lượt, bạn chọn **một đoạn liên tiếp gồm `k` hộp cùng màu** (`k >= 1`) và bỏ chúng đi, được **`k * k` điểm**. Sau khi bỏ, hai phần còn lại **khép sát vào nhau** (nên các hộp cùng màu ở hai bên có thể trở thành liền kề). Lặp đến khi hết hộp. Trả về **tổng điểm lớn nhất**.

**Ví dụ 1**

```text
Input:  boxes = [1,3,2,2,2,3,4,3,1]
Output: 23
Giải thích:
[1, 3, 2, 2, 2, 3, 4, 3, 1]
----> [1, 3, 3, 4, 3, 1]   (bỏ 3 hộp màu 2: 3*3 = 9 điểm)
----> [1, 3, 3, 3, 1]      (bỏ 1 hộp màu 4: 1*1 = 1 điểm)
----> [1, 1]               (bỏ 3 hộp màu 3: 3*3 = 9 điểm)
----> []                   (bỏ 2 hộp màu 1: 2*2 = 4 điểm)
Tổng: 9 + 1 + 9 + 4 = 23
```

**Ví dụ 2**

```text
Input:  boxes = [1,1,1]
Output: 9
```

**Ví dụ 3**

```text
Input:  boxes = [1]
Output: 1
```

**Ràng buộc**

- `1 <= boxes.length <= 100`
- `1 <= boxes[i] <= 100`

**Quan sát đầu tiên:** điểm là **bình phương**, nên gom được nhiều hộp cùng màu rồi mới bỏ một lần luôn có lợi hơn bỏ lẻ tẻ: `(a + b)^2 = a^2 + 2ab + b^2 > a^2 + b^2`. Toàn bộ bài toán xoay quanh câu hỏi: *nên bỏ những hộp nào ở giữa để các hộp cùng màu "gặp nhau"?*

In [1]:
def simulate(boxes, moves):
    """Mô phỏng: moves là danh sách (i, j) — bỏ đoạn boxes[i..j] (bao gồm 2 đầu) của dãy HIỆN TẠI."""
    cur = list(boxes)
    total = 0
    for i, j in moves:
        seg = cur[i:j + 1]
        assert len(set(seg)) == 1, f"đoạn {seg} không cùng màu"
        total += len(seg) ** 2
        cur = cur[:i] + cur[j + 1:]
        print(f"bỏ {seg} -> +{len(seg)**2:>2} điểm, còn {cur}")
    assert not cur
    return total

# Đúng chuỗi thao tác trong ví dụ 1
print("Tổng:", simulate([1, 3, 2, 2, 2, 3, 4, 3, 1], [(2, 4), (3, 3), (1, 3), (0, 1)]))

bỏ [2, 2, 2] -> + 9 điểm, còn [1, 3, 3, 4, 3, 1]
bỏ [4] -> + 1 điểm, còn [1, 3, 3, 3, 1]
bỏ [3, 3, 3] -> + 9 điểm, còn [1, 1]
bỏ [1, 1] -> + 4 điểm, còn []
Tổng: 23


## 3. Kiến thức nền

### 3.1. Đệ quy có nhớ với `functools.lru_cache`

Khi một hàm đệ quy được gọi lại nhiều lần với **cùng tham số**, ta lưu kết quả lại để lần sau trả ngay. `@lru_cache(maxsize=None)` (hoặc `@cache` từ Python 3.9) làm việc đó tự động. Điều kiện: mọi tham số phải **hashable** (số, chuỗi, tuple... — không phải list).

In [2]:
from functools import lru_cache

calls = 0
@lru_cache(maxsize=None)
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(30), "| số lần thân hàm thực sự chạy:", calls)   # chỉ 31 lần thay vì ~2.7 triệu
print(fib.cache_info())

832040 | số lần thân hàm thực sự chạy: 31
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


### 3.2. Tuple làm khóa và cắt ghép tuple

Brute force sẽ dùng **cả dãy hộp hiện tại** làm trạng thái. List không hash được, nên ta đổi sang `tuple`. Cắt ghép tuple giống list: `t[:i] + t[j:]` tạo tuple mới đã bỏ đoạn `t[i:j]`.

In [3]:
t = (1, 3, 2, 2, 2, 3)
print(t[:2] + t[5:])          # bỏ đoạn chỉ số 2..4
d = {t: "đã tính"}
print(d[(1, 3, 2, 2, 2, 3)])  # tra cứu bằng tuple có cùng nội dung
try:
    {[1, 2]: 0}
except TypeError as e:
    print("list không làm khóa được:", e)

(1, 3, 3)
đã tính
list không làm khóa được: unhashable type: 'list'


### 3.3. List 3 chiều — tránh bẫy `[[0]*n]*n`

DP bottom-up cần mảng `dp[l][r][k]`. Phải tạo bằng **list comprehension lồng nhau**; nếu nhân list (`[[0]*n]*n`) thì các hàng là **cùng một object**, sửa một chỗ thì đổi cả cột.

In [4]:
n = 3
bad = [[0] * n] * n
bad[0][0] = 7
print("sai :", bad)

good = [[[0] * n for _ in range(n)] for _ in range(n)]
good[0][0][0] = 7
print("đúng:", good[0], "| good[1][0][0] =", good[1][0][0])

sai : [[7, 0, 0], [7, 0, 0], [7, 0, 0]]
đúng: [[7, 0, 0], [0, 0, 0], [0, 0, 0]] | good[1][0][0] = 0


### 3.4. Nén dãy theo nhóm (run-length encoding) với `itertools.groupby`

`groupby` gom các phần tử **liền kề bằng nhau**. Ta sẽ dùng nó để biến `[1,3,2,2,2,3,4,3,1]` thành các cặp `(màu, số lượng)`.

In [5]:
from itertools import groupby
boxes = [1, 3, 2, 2, 2, 3, 4, 3, 1]
groups = [(color, len(list(g))) for color, g in groupby(boxes)]
print(groups)

[(1, 1), (3, 1), (2, 3), (3, 1), (4, 1), (3, 1), (1, 1)]


### 3.5. `sys.setrecursionlimit`

Đệ quy sâu (ví dụ 100 hộp, mỗi tầng giảm 1) có thể vượt giới hạn mặc định (~1000 khung). Với bài này độ sâu chỉ cỡ `n`, nhưng thói quen tốt là nâng giới hạn khi viết đệ quy trên input dài.

In [6]:
import sys
print("giới hạn mặc định:", sys.getrecursionlimit())
sys.setrecursionlimit(10000)
print("sau khi nâng     :", sys.getrecursionlimit())

giới hạn mặc định: 1000
sau khi nâng     : 10000


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Tính tay một ví dụ nhỏ: `[1, 2, 1]`

Có hai hướng đi:

| Thứ tự | Điểm |
|---|---|
| Bỏ `1` trái (1), bỏ `2` (1), bỏ `1` phải (1) | 3 |
| Bỏ `2` trước (1) → `[1, 1]` → bỏ cả hai (4) | **5** |

Bài học: **bỏ hộp ở giữa trước** để hai hộp cùng màu ở hai đầu gặp nhau.

### 4.2. Vì sao `dp[l][r]` không đủ?

Thử định nghĩa `f(l, r)` = điểm lớn nhất khi chỉ xét đoạn `boxes[l..r]`. Với `[1, 2, 1, 2, 1]`:

- Hộp `1` ở vị trí 0 có thể được bỏ **cùng** với hộp `1` ở vị trí 2 và 4. Nhưng khi đã chia thành các đoạn con, đoạn con `[1, 2, 1]` (vị trí 2..4) không biết rằng **bên ngoài nó còn một hộp `1` đang chờ để gộp**.
- Giá trị tốt nhất của đoạn con **phụ thuộc vào thứ bên ngoài** → `f(l, r)` không phải bài toán con độc lập.

### 4.3. Thêm thông tin: `k` hộp dính bên trái

Định nghĩa mới:

> `dp(l, r, k)` = điểm lớn nhất khi xét đoạn `boxes[l..r]`, **biết rằng bên trái `boxes[l]` đang có thêm `k` hộp cùng màu với `boxes[l]` đã dính sát vào** (các hộp ở giữa đã bị bỏ trước đó).

Khi đó hộp `boxes[l]` thực chất là một khối `k + 1` hộp. Ta có hai lựa chọn cho khối này:

1. **Bỏ ngay** khối `k + 1` hộp: được `(k+1)^2`, phần còn lại là `dp(l+1, r, 0)`.
2. **Giữ lại** để gộp với một hộp cùng màu `boxes[m]` ở xa hơn (`l < m <= r`, `boxes[m] == boxes[l]`): trước hết phải dọn sạch phần ở giữa `boxes[l+1..m-1]` (độc lập, được `dp(l+1, m-1, 0)`), rồi khối `k+1` hộp dính vào `boxes[m]` → `dp(m, r, k+1)`.

Hãy tính tay `[1, 2, 1]` với định nghĩa mới:

- `dp(0,2,0)`: hộp 0 màu 1.
  - Bỏ ngay: `1 + dp(1,2,0)`. `dp(1,2,0)` = đoạn `[2,1]` = 1 + 1 = 2. → tổng 3.
  - Gộp với `m = 2`: `dp(1,1,0) + dp(2,2,1)` = (bỏ `[2]`: 1) + (khối 2 hộp màu 1: 4) = **5**.
- Kết quả `max(3, 5) = 5`. Khớp với bảng ở 4.1.

In [7]:
# Liệt kê mọi thứ tự bỏ (chỉ bỏ các nhóm liền kề cực đại) cho [1, 2, 1] để thấy 5 là lớn nhất
def all_orders(state, score=0, path=()):
    if not state:
        yield score, path
        return
    i, n = 0, len(state)
    while i < n:
        j = i
        while j < n and state[j] == state[i]:
            j += 1
        yield from all_orders(state[:i] + state[j:], score + (j - i) ** 2, path + (state[i:j],))
        i = j

for score, path in sorted(all_orders((1, 2, 1)), reverse=True):
    print(score, path)

5 ((2,), (1, 1))
3 ((1,), (2,), (1,))
3 ((1,), (2,), (1,))
3 ((1,), (1,), (2,))
3 ((1,), (1,), (2,))


## 5. Cách 1 — Brute Force: vét cạn có nhớ trên cả dãy

**Ý tưởng.** Trạng thái = **dãy hộp còn lại** (tuple). Ở mỗi trạng thái, thử bỏ **mọi đoạn liên tiếp cùng màu** `state[i..j]`, đệ quy trên dãy còn lại, lấy max. Dùng `lru_cache` để không tính lại cùng một dãy.

Đây là cách "dịch nguyên văn đề bài" — chắc chắn đúng, dùng làm **chuẩn đối chiếu**.

In [8]:
from functools import lru_cache

def remove_boxes_brute(boxes):
    @lru_cache(maxsize=None)
    def best(state):
        if not state:
            return 0
        res = 0
        n = len(state)
        for i in range(n):
            j = i
            # thử mọi đoạn state[i..j] cùng màu (không chỉ nhóm cực đại — đúng nguyên văn đề)
            while j < n and state[j] == state[i]:
                cnt = j - i + 1
                res = max(res, cnt * cnt + best(state[:i] + state[j + 1:]))
                j += 1
        return res
    return best(tuple(boxes))

print(remove_boxes_brute([1, 3, 2, 2, 2, 3, 4, 3, 1]))
assert remove_boxes_brute([1, 3, 2, 2, 2, 3, 4, 3, 1]) == 23
assert remove_boxes_brute([1, 1, 1]) == 9
assert remove_boxes_brute([1]) == 1
assert remove_boxes_brute([1, 2, 1]) == 5

23


**Độ phức tạp.** Số dãy con có thể xuất hiện (mỗi dãy là một **dãy con không liên tiếp** của dãy gốc) lên tới `2^n` trạng thái, mỗi trạng thái thử `O(n^2)` đoạn và tạo tuple `O(n)` → khoảng `O(2^n · n^3)`. Với `n = 100` là bất khả thi.

**Nút thắt:** trạng thái "cả dãy còn lại" quá chi tiết. Ta cần một trạng thái **nhỏ gọn** (mô tả bằng vài số nguyên) mà vẫn đủ thông tin để quyết định.

In [9]:
import time, random
random.seed(1)
for n in [8, 12, 16]:
    arr = [random.randint(1, 3) for _ in range(n)]
    t = time.perf_counter()
    remove_boxes_brute(arr)
    print(f"n={n:>2}: {time.perf_counter() - t:.3f}s")

n= 8: 0.000s
n=12: 0.003s
n=16: 0.031s


## 6. Bước đột phá

**Insight 1 — Luôn có một hộp "chốt" để quyết định.** Xét hộp **trái nhất** `boxes[l]` của đoạn. Trong lời giải tối ưu, hộp này sẽ bị bỏ ở một lượt nào đó, **cùng với một tập các hộp cùng màu** nằm bên phải nó (có thể là tập rỗng). Gọi `m` là hộp **gần nhất** bên phải được bỏ cùng `boxes[l]`.

- Mọi hộp ở giữa `l` và `m` phải bị bỏ **trước** (để `l` và `m` liền kề) và chúng **không thể** gộp với gì ngoài đoạn `(l, m)` — vì chúng bị kẹp giữa hai hộp sẽ bỏ sau. Nên phần giữa là bài toán con **độc lập**: `dp(l+1, m-1, 0)`.
- Sau đó `boxes[l]` (cùng `k` hộp đang dính trước nó) dính vào `boxes[m]`: trạng thái `dp(m, r, k+1)`.

**Insight 2 — Trạng thái đủ thông tin.** Bên ngoài đoạn `[l, r]`, thứ duy nhất ảnh hưởng tới đoạn là "**có bao nhiêu hộp cùng màu `boxes[l]` đang chờ gộp ở bên trái**". Mọi thứ khác bên ngoài hoặc đã bị bỏ, hoặc sẽ được xử lý sau đoạn này một cách độc lập. → Thêm chiều `k` là đủ.

**Insight 3 — Gộp nhóm liền kề ngay từ đầu.** Nếu `boxes[l+1] == boxes[l]` thì gộp luôn chúng là không thiệt (chúng đã liền kề; tách ra chỉ làm mất điểm bình phương). Ta tăng `l` và `k` cho tới khi hộp tiếp theo khác màu → giảm mạnh số trạng thái.

**Quy tắc chuyển trạng thái:**

```text
dp(l, r, k) = 0                                     nếu l > r
dp(l, r, k) = max( (k+1)^2 + dp(l+1, r, 0),
                   max over m in (l, r], boxes[m] == boxes[l]:
                        dp(l+1, m-1, 0) + dp(m, r, k+1) )
Đáp án = dp(0, n-1, 0)
```

In [10]:
# Minh họa insight 2: cùng đoạn [1, 2, 1] nhưng khác k thì đáp án khác hẳn
from functools import lru_cache

def dp_demo(boxes):
    @lru_cache(maxsize=None)
    def dp(l, r, k):
        if l > r:
            return 0
        res = (k + 1) ** 2 + dp(l + 1, r, 0)
        for m in range(l + 1, r + 1):
            if boxes[m] == boxes[l]:
                res = max(res, dp(l + 1, m - 1, 0) + dp(m, r, k + 1))
        return res
    return dp

dp = dp_demo([1, 2, 1])
for k in range(4):
    print(f"dp(0, 2, k={k}) = {dp(0, 2, k)}   (tức là có {k} hộp màu 1 đang chờ ở bên trái)")

dp(0, 2, k=0) = 5   (tức là có 0 hộp màu 1 đang chờ ở bên trái)
dp(0, 2, k=1) = 10   (tức là có 1 hộp màu 1 đang chờ ở bên trái)
dp(0, 2, k=2) = 17   (tức là có 2 hộp màu 1 đang chờ ở bên trái)
dp(0, 2, k=3) = 26   (tức là có 3 hộp màu 1 đang chờ ở bên trái)


Nhận xét: `dp(0,2,k) = (k+2)^2 + 1` — hộp `2` bỏ riêng (1 điểm), hai hộp `1` gộp với `k` hộp chờ → `(k+2)^2`. Đoạn con "biết" được bên ngoài nhờ `k`.

## 7. Các cách tiếp theo

Tổng cộng notebook trình bày **4 cách**:

| # | Cách | Trạng thái |
|---|---|---|
| 1 | Brute force có nhớ (mục 5) | cả dãy còn lại |
| 2 | Đệ quy có nhớ `dp(l, r, k)` + gộp nhóm liền kề | 3 số nguyên |
| 3 | DP bảng 3 chiều (bottom-up) | mảng `dp[l][r][k]` |
| 4 | DP trên nhóm đã nén (run-length) | `dp(i, j, extra)` trên nhóm |

### 7.1. Cách 2 — Đệ quy có nhớ `dp(l, r, k)`

**Giải thích.** Cài đúng công thức ở mục 6, thêm bước gộp nhóm liền kề (Insight 3). Lưu ý khi gộp: mỗi lần dịch `l` sang phải một ô, hộp cũ ở vị trí `l` nhập vào khối "đang dính" → `k += 1`.

Code demo có in số trạng thái để thấy hiệu quả của bước gộp:

In [11]:
def remove_boxes_memo(boxes, merge=True):
    n = len(boxes)

    @lru_cache(maxsize=None)
    def dp(l, r, k):
        if l > r:
            return 0
        if merge:
            # Insight 3: hộp kế bên cùng màu -> gộp luôn vào khối đang dính
            while l < r and boxes[l + 1] == boxes[l]:
                l += 1
                k += 1
        # Lựa chọn 1: bỏ ngay khối (k+1) hộp màu boxes[l]
        res = (k + 1) ** 2 + dp(l + 1, r, 0)
        # Lựa chọn 2: giữ lại, gộp với hộp boxes[m] cùng màu ở xa hơn
        for m in range(l + 1, r + 1):
            if boxes[m] == boxes[l]:
                res = max(res, dp(l + 1, m - 1, 0) + dp(m, r, k + 1))
        return res

    ans = dp(0, n - 1, 0)
    return ans, dp.cache_info().currsize

for merge in (False, True):
    ans, states = remove_boxes_memo([1, 3, 2, 2, 2, 3, 4, 3, 1], merge)
    print(f"merge={merge}: đáp án {ans}, số trạng thái đã lưu {states}")

random.seed(7)
arr = [random.randint(1, 2) for _ in range(40)]
for merge in (False, True):
    t = time.perf_counter()
    ans, states = remove_boxes_memo(arr, merge)
    print(f"n=40, merge={merge}: đáp án {ans}, trạng thái {states}, {time.perf_counter()-t:.3f}s")

merge=False: đáp án 23, số trạng thái đã lưu 50
merge=True: đáp án 23, số trạng thái đã lưu 27
n=40, merge=False: đáp án 802, trạng thái 6221, 0.015s
n=40, merge=True: đáp án 802, trạng thái 4614, 0.008s


**Đọc code từng dòng:**

- `if l > r: return 0` — đoạn rỗng, không còn hộp. Lưu ý: khi đoạn rỗng thì `k` hộp đang dính **không được tính** ở đây; chúng luôn được tính ở bước "bỏ ngay" của hộp `l` mà chúng dính vào, nên không bị mất.
- Vòng `while` gộp nhóm: chỉ đổi biến cục bộ `l`, `k`; `lru_cache` vẫn lưu theo tham số **gốc** khi gọi — không sao vì kết quả của tham số gốc đúng bằng kết quả sau khi gộp.
- `res = (k+1)**2 + dp(l+1, r, 0)` — bỏ khối ngay; phần còn lại không có gì dính (hộp `l+1` khác màu với khối vừa bỏ, và khối đã biến mất).
- Vòng `for m` — thử mọi "bạn gộp" `m` cùng màu. Phần giữa `dp(l+1, m-1, 0)` độc lập; phần sau `dp(m, r, k+1)` mang theo khối đã lớn thêm 1.

**Code nộp LeetCode (Cách 2):**

In [12]:
from functools import lru_cache
from typing import List

class SolutionMemo:
    def removeBoxes(self, boxes: List[int]) -> int:
        n = len(boxes)

        @lru_cache(maxsize=None)
        def dp(l: int, r: int, k: int) -> int:
            if l > r:
                return 0
            while l < r and boxes[l + 1] == boxes[l]:
                l += 1
                k += 1
            res = (k + 1) * (k + 1) + dp(l + 1, r, 0)
            for m in range(l + 1, r + 1):
                if boxes[m] == boxes[l]:
                    res = max(res, dp(l + 1, m - 1, 0) + dp(m, r, k + 1))
            return res

        return dp(0, n - 1, 0)

assert SolutionMemo().removeBoxes([1, 3, 2, 2, 2, 3, 4, 3, 1]) == 23
assert SolutionMemo().removeBoxes([1, 1, 1]) == 9
print("SolutionMemo OK")

SolutionMemo OK


**Độ phức tạp Cách 2.** Có tối đa `n · n · n` trạng thái (`l`, `r`, `k` đều `< n`), mỗi trạng thái duyệt `O(n)` giá trị `m` → **Time `O(n^4)`** trong trường hợp xấu nhất, **Space `O(n^3)`**. Thực tế số trạng thái **thực sự được gọi** nhỏ hơn nhiều (vì `k` chỉ tăng khi có hộp cùng màu) — với `n = 100` chạy được.

### 7.2. Cách 3 — DP bảng 3 chiều (bottom-up)

**Giải thích.** Cùng công thức nhưng điền bảng theo thứ tự sao cho mọi giá trị cần dùng đã có sẵn:

- `dp(l, r, k)` cần `dp(l+1, ·, ·)` và `dp(m, ·, ·)` với `m > l` → duyệt **`l` từ lớn xuống nhỏ**.
- Trong cùng `l`, chỉ cần các `l` lớn hơn nên thứ tự `r`, `k` tùy ý.
- `k` tối đa bằng `l` (bên trái hộp `l` chỉ có `l` hộp).
- Đoạn rỗng (`l > r`) có giá trị 0: chỉ cần để mảng mặc định 0 và cấp phát kích thước `n + 1` để chỉ số `l + 1 = n` không tràn.

In [13]:
def remove_boxes_bottom_up(boxes):
    n = len(boxes)
    # dp[l][r][k], kích thước (n+1) x (n+1) x (n+1); ô có l > r mặc định 0
    dp = [[[0] * (n + 1) for _ in range(n + 1)] for _ in range(n + 1)]
    for l in range(n - 1, -1, -1):
        for r in range(l, n):
            for k in range(l + 1):                       # 0..l hộp có thể dính bên trái
                res = (k + 1) ** 2 + dp[l + 1][r][0]
                for m in range(l + 1, r + 1):
                    if boxes[m] == boxes[l]:
                        res = max(res, dp[l + 1][m - 1][0] + dp[m][r][k + 1])
                dp[l][r][k] = res
    return dp[0][n - 1][0] if n else 0

print(remove_boxes_bottom_up([1, 3, 2, 2, 2, 3, 4, 3, 1]))
assert remove_boxes_bottom_up([1, 3, 2, 2, 2, 3, 4, 3, 1]) == 23
assert remove_boxes_bottom_up([1, 2, 1]) == 5

23


**Kiểm tra chỉ số biên:**

- `dp[l+1][m-1][0]` khi `m = l+1` → `dp[l+1][l][0]`: đoạn rỗng (`l+1 > l`), ô chưa từng được ghi → 0. Đúng.
- `dp[m][r][k+1]`: `m > l` nên đã được tính ở vòng `l` trước; `k + 1 <= l + 1 <= m` nằm trong phạm vi `k` hợp lệ của `m`.

**Code nộp LeetCode (Cách 3):** bottom-up luôn tính **mọi** trạng thái `O(n^3)` × `O(n)` vòng `m` — với `n = 100` khoảng `10^8 / 24` phép, chậm trong Python; nên coi đây là cách để hiểu thứ tự tính, còn nộp thì Cách 2 hoặc 4 an toàn hơn. Ta tối ưu nhẹ bằng cách chỉ duyệt các `m` cùng màu (danh sách tính trước):

In [14]:
class SolutionBottomUp:
    def removeBoxes(self, boxes: List[int]) -> int:
        n = len(boxes)
        # same[l] = các vị trí m > l có cùng màu với boxes[l]
        same = [[m for m in range(l + 1, n) if boxes[m] == boxes[l]] for l in range(n)]
        dp = [[[0] * (n + 1) for _ in range(n + 1)] for _ in range(n + 1)]
        for l in range(n - 1, -1, -1):
            nxt = dp[l + 1]
            for r in range(l, n):
                cand = [m for m in same[l] if m <= r]
                row = dp[l][r]
                for k in range(l + 1):
                    res = (k + 1) * (k + 1) + nxt[r][0]
                    for m in cand:
                        v = nxt[m - 1][0] + dp[m][r][k + 1]
                        if v > res:
                            res = v
                    row[k] = res
        return dp[0][n - 1][0]

assert SolutionBottomUp().removeBoxes([1, 3, 2, 2, 2, 3, 4, 3, 1]) == 23
assert SolutionBottomUp().removeBoxes([1]) == 1
print("SolutionBottomUp OK")

SolutionBottomUp OK


**Độ phức tạp Cách 3:** Time `O(n^4)` (luôn đạt cận vì tính mọi ô), Space `O(n^3)` ≈ 1 triệu ô với `n = 100`.

### 7.3. Cách 4 — DP trên nhóm đã nén (run-length)

**Giải thích.** Insight 3 nói các hộp liền kề cùng màu luôn đi cùng nhau. Vậy nén trước: `[1,3,2,2,2,3,4,3,1]` → nhóm `(1,1) (3,1) (2,3) (3,1) (4,1) (3,1) (1,1)`. Gọi `color[i]`, `cnt[i]` là màu và số hộp của nhóm `i`.

> `g(i, j, extra)` = điểm lớn nhất cho các nhóm `i..j`, biết có thêm `extra` **hộp** cùng màu `color[i]` dính bên trái nhóm `i`.

```text
g(i, j, extra) = max( (cnt[i] + extra)^2 + g(i+1, j, 0),
                      max over m in (i, j], color[m] == color[i]:
                           g(i+1, m-1, 0) + g(m, j, extra + cnt[i]) )
```

Lợi ích: số nhóm `g <= n`, và nhóm liền kề luôn **khác màu** nên ứng viên `m` thường ít hơn. Đây là cùng một ý tưởng, chỉ là chạy trên dữ liệu gọn hơn.

In [15]:
def remove_boxes_groups(boxes):
    color, cnt = [], []
    for c, g in groupby(boxes):
        color.append(c)
        cnt.append(len(list(g)))
    G = len(color)

    @lru_cache(maxsize=None)
    def g(i, j, extra):
        if i > j:
            return 0
        res = (cnt[i] + extra) ** 2 + g(i + 1, j, 0)
        for m in range(i + 2, j + 1):           # m = i+1 luôn khác màu nên bắt đầu từ i+2
            if color[m] == color[i]:
                res = max(res, g(i + 1, m - 1, 0) + g(m, j, extra + cnt[i]))
        return res

    return g(0, G - 1, 0)

print(remove_boxes_groups([1, 3, 2, 2, 2, 3, 4, 3, 1]))
assert remove_boxes_groups([1, 3, 2, 2, 2, 3, 4, 3, 1]) == 23
assert remove_boxes_groups([2, 2, 2, 2]) == 16

23


**Đọc code:**

- `range(i + 2, j + 1)`: sau khi nén, nhóm `i+1` chắc chắn khác màu nhóm `i`, nên bỏ qua.
- `extra + cnt[i]`: khi nhóm `i` dính vào nhóm `m`, số hộp dính tăng **bằng kích thước nhóm** chứ không phải +1 như Cách 2.
- `extra` có thể lớn tới `n`, nhưng chỉ nhận các giá trị là tổng kích thước một số nhóm cùng màu.

**Code nộp LeetCode (Cách 4):**

In [16]:
from itertools import groupby

class SolutionGroups:
    def removeBoxes(self, boxes: List[int]) -> int:
        color, cnt = [], []
        for c, grp in groupby(boxes):
            color.append(c)
            cnt.append(len(list(grp)))

        @lru_cache(maxsize=None)
        def g(i: int, j: int, extra: int) -> int:
            if i > j:
                return 0
            res = (cnt[i] + extra) ** 2 + g(i + 1, j, 0)
            for m in range(i + 2, j + 1):
                if color[m] == color[i]:
                    res = max(res, g(i + 1, m - 1, 0) + g(m, j, extra + cnt[i]))
            return res

        return g(0, len(color) - 1, 0)

assert SolutionGroups().removeBoxes([1, 3, 2, 2, 2, 3, 4, 3, 1]) == 23
assert SolutionGroups().removeBoxes([1, 1, 1]) == 9
print("SolutionGroups OK")

SolutionGroups OK


**Độ phức tạp Cách 4:** với `G` nhóm: tối đa `G · G · (n+1)` trạng thái, mỗi trạng thái `O(G)` → Time `O(G^3 · n)` ≤ `O(n^4)`, Space `O(G^2 · n)`. Khi dãy có nhiều đoạn lặp màu, `G` nhỏ hơn hẳn `n`.

## 8. TRACE từng bước

### 8.1. Trace cây lời gọi của Cách 2 trên `[1, 2, 1]`

Bảng tóm tắt các trạng thái (không gộp nhóm để thấy rõ):

| Lời gọi | Lựa chọn "bỏ ngay" | Lựa chọn "gộp với m" | Kết quả |
|---|---|---|---|
| `dp(2,2,1)` | `(1+1)^2 + dp(3,2,0) = 4` | — | 4 |
| `dp(2,2,0)` | `1 + 0 = 1` | — | 1 |
| `dp(1,1,0)` | `1 + 0 = 1` | — | 1 |
| `dp(1,2,0)` | `1 + dp(2,2,0) = 2` | (không có hộp màu 2 nào phía sau) | 2 |
| `dp(0,2,0)` | `1 + dp(1,2,0) = 3` | `m=2`: `dp(1,1,0) + dp(2,2,1) = 1 + 4 = 5` | **5** |

Hàm `trace_dp` dưới đây in mỗi lời gọi (thụt lề theo độ sâu), đánh dấu khi lấy từ bộ nhớ.

In [17]:
def trace_dp(boxes, merge=False):
    memo = {}

    def dp(l, r, k, depth=0):
        pad = "    " * depth
        key = (l, r, k)
        if l > r:
            return 0
        if key in memo:
            print(f"{pad}dp{key} = {memo[key]}  (lấy từ memo)")
            return memo[key]
        print(f"{pad}dp{key}: đoạn {boxes[l:r+1]}, có {k} hộp màu {boxes[l]} dính bên trái")
        L, K = l, k
        if merge:
            while L < r and boxes[L + 1] == boxes[L]:
                L += 1; K += 1
            if L != l:
                print(f"{pad}  gộp nhóm liền kề -> l={L}, k={K}")
        res = (K + 1) ** 2 + dp(L + 1, r, 0, depth + 1)
        print(f"{pad}  bỏ ngay khối {K+1} hộp: {res}")
        for m in range(L + 1, r + 1):
            if boxes[m] == boxes[L]:
                v = dp(L + 1, m - 1, 0, depth + 1) + dp(m, r, K + 1, depth + 1)
                print(f"{pad}  gộp với m={m}: {v}")
                res = max(res, v)
        memo[key] = res
        print(f"{pad}=> dp{key} = {res}")
        return res

    return dp(0, len(boxes) - 1, 0)

print("Kết quả:", trace_dp([1, 2, 1]))

dp(0, 2, 0): đoạn [1, 2, 1], có 0 hộp màu 1 dính bên trái
    dp(1, 2, 0): đoạn [2, 1], có 0 hộp màu 2 dính bên trái
        dp(2, 2, 0): đoạn [1], có 0 hộp màu 1 dính bên trái
          bỏ ngay khối 1 hộp: 1
        => dp(2, 2, 0) = 1
      bỏ ngay khối 1 hộp: 2
    => dp(1, 2, 0) = 2
  bỏ ngay khối 1 hộp: 3
    dp(1, 1, 0): đoạn [2], có 0 hộp màu 2 dính bên trái
      bỏ ngay khối 1 hộp: 1
    => dp(1, 1, 0) = 1
    dp(2, 2, 1): đoạn [1], có 1 hộp màu 1 dính bên trái
      bỏ ngay khối 2 hộp: 4
    => dp(2, 2, 1) = 4
  gộp với m=2: 5
=> dp(0, 2, 0) = 5
Kết quả: 5


### 8.2. Trace có gộp nhóm trên `[2, 2, 1, 2]`

Tính tay: tốt nhất là bỏ `1` (1 điểm) rồi bỏ cả ba hộp `2` (9 điểm) = **10**. Xem cách DP tìm ra: ở `dp(0,3,0)`, bước gộp đưa `l` lên 1 với `k = 1` (hai hộp `2` đầu đã thành một khối), rồi thử gộp với `m = 3`.

In [18]:
print("Kết quả:", trace_dp([2, 2, 1, 2], merge=True))

dp(0, 3, 0): đoạn [2, 2, 1, 2], có 0 hộp màu 2 dính bên trái
  gộp nhóm liền kề -> l=1, k=1
    dp(2, 3, 0): đoạn [1, 2], có 0 hộp màu 1 dính bên trái
        dp(3, 3, 0): đoạn [2], có 0 hộp màu 2 dính bên trái
          bỏ ngay khối 1 hộp: 1
        => dp(3, 3, 0) = 1
      bỏ ngay khối 1 hộp: 2
    => dp(2, 3, 0) = 2
  bỏ ngay khối 2 hộp: 6
    dp(2, 2, 0): đoạn [1], có 0 hộp màu 1 dính bên trái
      bỏ ngay khối 1 hộp: 1
    => dp(2, 2, 0) = 1
    dp(3, 3, 2): đoạn [2], có 2 hộp màu 2 dính bên trái
      bỏ ngay khối 3 hộp: 9
    => dp(3, 3, 2) = 9
  gộp với m=3: 10
=> dp(0, 3, 0) = 10
Kết quả: 10


### 8.3. Bảng `dp[l][r][0]` của Cách 3 cho ví dụ 1

In phần `k = 0` của bảng bottom-up: ô `(l, r)` là điểm tốt nhất của đoạn `boxes[l..r]` khi không có gì dính. Đường chéo chính toàn số 1 (một hộp đơn), góc trên phải là đáp án 23.

In [19]:
def trace_table(boxes):
    n = len(boxes)
    dp = [[[0] * (n + 1) for _ in range(n + 1)] for _ in range(n + 1)]
    for l in range(n - 1, -1, -1):
        for r in range(l, n):
            for k in range(l + 1):
                res = (k + 1) ** 2 + dp[l + 1][r][0]
                for m in range(l + 1, r + 1):
                    if boxes[m] == boxes[l]:
                        res = max(res, dp[l + 1][m - 1][0] + dp[m][r][k + 1])
                dp[l][r][k] = res
    print("boxes:", boxes)
    print("l\\r " + "".join(f"{r:>4}" for r in range(n)))
    for l in range(n):
        print(f"{l:>3} " + "".join(f"{dp[l][r][0]:>4}" if r >= l else "   ." for r in range(n)))
    return dp[0][n - 1][0]

trace_table([1, 3, 2, 2, 2, 3, 4, 3, 1])

boxes: [1, 3, 2, 2, 2, 3, 4, 3, 1]
l\r    0   1   2   3   4   5   6   7   8
  0    1   2   3   6  11  14  15  20  23
  1    .   1   2   5  10  13  14  19  20
  2    .   .   1   4   9  10  11  14  15
  3    .   .   .   1   4   5   6   9  10
  4    .   .   .   .   1   2   3   6   7
  5    .   .   .   .   .   1   2   5   6
  6    .   .   .   .   .   .   1   2   3
  7    .   .   .   .   .   .   .   1   2
  8    .   .   .   .   .   .   .   .   1


23

Đọc bảng: ô `(1, 7)` là đoạn `[3,2,2,2,3,4,3]` → 9 (ba hộp 2) + 1 (hộp 4) + 9 (ba hộp 3) = 19; cộng thêm hai hộp `1` ở hai đầu gộp lại (4) → 23 ở ô `(0, 8)`.

## 9. Code nộp LeetCode

Chọn Cách 2 (đệ quy có nhớ + gộp nhóm): ngắn, nhanh nhờ chỉ thăm trạng thái cần thiết. Dùng `@lru_cache` trên hàm lồng trong method để cache **không bị chia sẻ** giữa các test.

In [20]:
from functools import lru_cache
from typing import List

class Solution:
    def removeBoxes(self, boxes: List[int]) -> int:
        n = len(boxes)

        @lru_cache(maxsize=None)
        def dp(l: int, r: int, k: int) -> int:
            # k = số hộp cùng màu boxes[l] đang dính sát bên trái đoạn [l, r]
            if l > r:
                return 0
            while l < r and boxes[l + 1] == boxes[l]:   # gộp nhóm liền kề
                l += 1
                k += 1
            best = (k + 1) * (k + 1) + dp(l + 1, r, 0)  # bỏ ngay khối k+1 hộp
            for m in range(l + 1, r + 1):               # hoặc gộp với boxes[m] cùng màu
                if boxes[m] == boxes[l]:
                    best = max(best, dp(l + 1, m - 1, 0) + dp(m, r, k + 1))
            return best

        return dp(0, n - 1, 0)

sol = Solution()
assert sol.removeBoxes([1, 3, 2, 2, 2, 3, 4, 3, 1]) == 23
assert sol.removeBoxes([1, 1, 1]) == 9
assert sol.removeBoxes([1]) == 1
print("Solution OK")

Solution OK


In [21]:
# Thử với n = 100 (trường hợp lớn nhất) để chắc chắn thời gian chấp nhận được
random.seed(2024)
big = [random.randint(1, 3) for _ in range(100)]
t = time.perf_counter()
print("n=100, 3 màu:", Solution().removeBoxes(big), f"{time.perf_counter()-t:.2f}s")
t = time.perf_counter()
print("n=100, 1 màu:", Solution().removeBoxes([5] * 100), f"{time.perf_counter()-t:.2f}s")

n=100, 3 màu: 1536 0.13s
n=100, 1 màu: 10000 0.00s


## 10. Vì sao thuật toán đúng

Chứng minh công thức `dp(l, r, k)` theo hai chiều.

**(a) Mọi giá trị DP tính ra đều đạt được (không "ăn gian").** Mỗi lựa chọn tương ứng một chuỗi thao tác hợp lệ:
- "Bỏ ngay": bỏ khối `k+1` hộp (chúng liền kề thật vì `k` hộp kia đã dính sát) rồi xử lý phần còn lại.
- "Gộp với `m`": trước tiên bỏ hết `boxes[l+1..m-1]` theo cách tối ưu của đoạn đó (các thao tác chỉ đụng vào đoạn giữa nên hợp lệ), khi đó khối `k+1` hộp liền kề `boxes[m]` → đúng tình huống `dp(m, r, k+1)`.

**(b) Lời giải tối ưu luôn được DP xét tới.** Xét lời giải tối ưu bất kỳ cho tình huống `(l, r, k)`. Khối chứa `boxes[l]` bị bỏ cùng một tập hộp cùng màu bên phải. Nếu tập rỗng → đúng lựa chọn "bỏ ngay". Ngược lại, gọi `m` là hộp **gần nhất** trong tập đó:
- Các hộp `l+1..m-1` phải bị bỏ trước khi `l` và `m` gặp nhau, và chúng không thể bỏ cùng hộp nào ngoài khoảng (vì `l` và `m` còn đó chặn hai đầu cho đến khi chúng bị bỏ hết). Vậy điểm từ chúng ≤ `dp(l+1, m-1, 0)`.
- Phần còn lại tương đương tình huống "đoạn `[m, r]` có `k+1` hộp cùng màu dính bên trái", nên điểm ≤ `dp(m, r, k+1)`.

Cộng lại: tối ưu ≤ giá trị DP xét ở lựa chọn `m`. Kết hợp (a) → DP = tối ưu. Quy nạp theo độ dài đoạn hoàn tất chứng minh.

**Bước gộp nhóm** an toàn vì: nếu `boxes[l+1] == boxes[l]`, luôn có lời giải tối ưu bỏ chúng cùng lượt (tách ra chỉ làm điểm bình phương nhỏ hơn hoặc bằng), nên coi `boxes[l]` là một hộp dính thêm của `boxes[l+1]`.

## 11. Phân tích Time/Space

| Cách | Số trạng thái | Việc mỗi trạng thái | Time | Space |
|---|---|---|---|---|
| 1. Brute force tuple | ≤ `2^n` | `O(n^2)` đoạn × tạo tuple `O(n)` | `O(2^n · n^3)` | `O(2^n · n)` |
| 2. `dp(l,r,k)` có nhớ | ≤ `n^3` | `O(n)` | `O(n^4)` (thực tế ít hơn nhiều) | `O(n^3)` |
| 3. Bảng 3 chiều | đúng `~n^3/6` ô | `O(n)` | `Θ(n^4)` | `O(n^3)` |
| 4. Nhóm nén | ≤ `G^2 · n` | `O(G)` | `O(G^3 · n)` | `O(G^2 · n)` |

Với `n = 100`: `n^4 = 10^8` là cận trên lỏng; Cách 2/4 chỉ thăm phần nhỏ trạng thái nên chạy trong thời gian chấp nhận được trên LeetCode.

Đo nhanh để thấy số trạng thái thực sự được thăm:

In [22]:
for colors in (1, 2, 5, 20):
    random.seed(colors)
    arr = [random.randint(1, colors) for _ in range(60)]
    ans, states = remove_boxes_memo(arr)
    print(f"n=60, {colors:>2} màu: đáp án {ans:>5}, trạng thái thăm {states:>6} (so với n^3 = {60**3})")

n=60,  1 màu: đáp án  3600, trạng thái thăm      2 (so với n^3 = 216000)
n=60,  2 màu: đáp án  1512, trạng thái thăm  13220 (so với n^3 = 216000)


n=60,  5 màu: đáp án   484, trạng thái thăm   7539 (so với n^3 = 216000)
n=60, 20 màu: đáp án   134, trạng thái thăm   1715 (so với n^3 = 216000)


## 12. Bộ test

Gồm: ví dụ đề, edge case, và đối chiếu 4 cách với brute force trên input ngẫu nhiên có seed.

In [23]:
def run_all(boxes):
    return [
        remove_boxes_brute(boxes) if len(boxes) <= 12 else None,
        remove_boxes_memo(boxes)[0],
        remove_boxes_bottom_up(boxes),
        remove_boxes_groups(boxes),
        Solution().removeBoxes(boxes),
    ]

# Ví dụ đề
assert Solution().removeBoxes([1, 3, 2, 2, 2, 3, 4, 3, 1]) == 23
assert Solution().removeBoxes([1, 1, 1]) == 9
assert Solution().removeBoxes([1]) == 1

# Edge case
edge = {
    (1,): 1,                       # một hộp
    (7, 7, 7, 7, 7): 25,           # toàn một màu
    (1, 2, 3, 4): 4,               # toàn khác màu: mỗi hộp 1 điểm
    (1, 2, 1, 2, 1): 11,           # bỏ hai hộp 2 (1+1), gộp ba hộp 1 (9)
    (1, 2, 2, 1): 8,               # bỏ 2,2 (4) rồi 1,1 (4)
}
for arr, exp in edge.items():
    res = run_all(list(arr))
    assert all(x == exp for x in res if x is not None), (arr, res)
print("Edge cases OK")

Edge cases OK


In [24]:
# Đối chiếu ngẫu nhiên với brute force
random.seed(546)
cnt = 0
for _ in range(300):
    n = random.randint(1, 10)
    arr = [random.randint(1, random.randint(1, 4)) for _ in range(n)]
    res = run_all(arr)
    assert len(set(res)) == 1, (arr, res)
    cnt += 1
print(f"{cnt} test ngẫu nhiên: 5 cách khớp nhau")

# Test lớn hơn: chỉ đối chiếu các cách đa thức với nhau
for _ in range(20):
    n = random.randint(20, 40)
    arr = [random.randint(1, 3) for _ in range(n)]
    res = run_all(arr)[1:]
    assert len(set(res)) == 1, (arr, res)
print("20 test n=20..40: các cách DP khớp nhau")

300 test ngẫu nhiên: 5 cách khớp nhau
20 test n=20..40: các cách DP khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `dp[l][r]` hai chiều

**Code sai:** chia đoạn tại mọi điểm `m` và cộng hai nửa, giống bài "nhân ma trận".

In [25]:
def wrong_2d(boxes):
    @lru_cache(maxsize=None)
    def f(l, r):
        if l > r:
            return 0
        if all(b == boxes[l] for b in boxes[l:r + 1]):
            return (r - l + 1) ** 2
        return max(f(l, m) + f(m + 1, r) for m in range(l, r))
    return f(0, len(boxes) - 1)

print("dp 2 chiều:", wrong_2d([1, 2, 1]), "| đúng:", Solution().removeBoxes([1, 2, 1]))

dp 2 chiều: 3 | đúng: 5


**Vì sao sai:** chia đôi rồi cộng nghĩa là hai nửa **không bao giờ** gộp hộp với nhau, nên cách "bỏ hộp giữa để hai đầu gặp nhau" bị mất. Với `[1, 2, 1]`, hai hộp `1` luôn bị tách → chỉ được 3 thay vì 5.

**Code đúng:** thêm chiều `k` như `Solution` ở mục 9.

### Lỗi 2 — Tham lam "bỏ nhóm lớn nhất trước"

In [26]:
def greedy_largest(boxes):
    cur = list(boxes)
    total = 0
    while cur:
        # tìm nhóm liền kề dài nhất
        best_i, best_j, i = 0, 0, 0
        while i < len(cur):
            j = i
            while j + 1 < len(cur) and cur[j + 1] == cur[i]:
                j += 1
            if j - i > best_j - best_i:
                best_i, best_j = i, j
            i = j + 1
        total += (best_j - best_i + 1) ** 2
        cur = cur[:best_i] + cur[best_j + 1:]
    return total

# Để máy tự tìm phản ví dụ nhỏ nhất
random.seed(0)
found = None
for _ in range(5000):
    arr = [random.randint(1, 3) for _ in range(random.randint(1, 7))]
    if greedy_largest(arr) != Solution().removeBoxes(arr):
        if found is None or len(arr) < len(found):
            found = arr
assert found is not None
print("phản ví dụ:", found, "| tham lam:", greedy_largest(found), "| tối ưu:", Solution().removeBoxes(found))

phản ví dụ: [3, 2, 3] | tham lam: 3 | tối ưu: 5


**Vì sao sai:** bỏ nhóm lớn nhất ngay có thể **phá cơ hội** để nhóm đó (hoặc nhóm khác) gộp thêm hộp sau này. Bài này không có cấu trúc tham lam; phải xét mọi khả năng → DP.

**Code đúng:** DP ở mục 9.

### Lỗi 3 — Quên `k` khi gộp nhóm liền kề, hoặc tăng `l` mà không tăng `k`

In [27]:
def wrong_merge(boxes):
    @lru_cache(maxsize=None)
    def dp(l, r, k):
        if l > r:
            return 0
        while l < r and boxes[l + 1] == boxes[l]:
            l += 1                      # QUÊN k += 1
        res = (k + 1) ** 2 + dp(l + 1, r, 0)
        for m in range(l + 1, r + 1):
            if boxes[m] == boxes[l]:
                res = max(res, dp(l + 1, m - 1, 0) + dp(m, r, k + 1))
        return res
    return dp(0, len(boxes) - 1, 0)

print("sai:", wrong_merge([1, 1, 1]), "| đúng:", Solution().removeBoxes([1, 1, 1]))

sai: 1 | đúng: 9


**Vì sao sai:** dịch `l` qua một hộp cùng màu tức là hộp đó **trở thành một hộp dính** — nếu không tăng `k` thì hộp đó biến mất khỏi phép tính. `[1,1,1]` chỉ được 1 điểm.

**Code đúng:** `l += 1; k += 1` cùng nhau.

### Lỗi 4 — Cache toàn cục dùng chung giữa các test

In [28]:
# SAI: hàm dp ở cấp module tham chiếu biến toàn cục BOXES -> cache cũ còn dính khi đổi BOXES
BOXES = [1, 2, 1]
@lru_cache(maxsize=None)
def dp_global(l, r, k):
    if l > r:
        return 0
    res = (k + 1) ** 2 + dp_global(l + 1, r, 0)
    for m in range(l + 1, r + 1):
        if BOXES[m] == BOXES[l]:
            res = max(res, dp_global(l + 1, m - 1, 0) + dp_global(m, r, k + 1))
    return res

print("test 1:", dp_global(0, 2, 0))       # 5, đúng
BOXES = [3, 3, 3]
print("test 2:", dp_global(0, 2, 0), "| đúng phải là", Solution().removeBoxes([3, 3, 3]))

test 1: 5
test 2: 5 | đúng phải là 9


**Vì sao sai:** `lru_cache` nhớ theo `(l, r, k)` — không biết `BOXES` đã đổi, nên trả kết quả của test trước.

**Code đúng:** định nghĩa `dp` **bên trong** method (mỗi lần gọi tạo hàm + cache mới), hoặc gọi `dp_global.cache_clear()` trước mỗi test.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu "DP đoạn có thêm chiều":** bỏ/ghép phần tử làm hai phía **khép lại**, và điểm phụ thuộc vào việc các phần tử ở xa **gặp nhau** (ví dụ: Zuma, Strange Printer, Remove Boxes). Khi `dp[l][r]` không đủ, hãy hỏi: *"đoạn con cần biết thêm gì về bên ngoài?"* → đưa thông tin đó vào trạng thái.
- **Chọn phần tử "chốt" ở một đầu** (ở đây là hộp trái nhất) và hỏi "nó bị xử lý cùng với ai?" — một kỹ thuật chung cho DP đoạn.
- **Điểm bình phương / lồi** → gom càng nhiều càng tốt → gộp nhóm liền kề an toàn → nén run-length.
- **Đệ quy có nhớ thường thắng bottom-up** khi số trạng thái thực sự cần nhỏ hơn nhiều so với không gian trạng thái đầy đủ.
- **Kiểm chứng bằng brute force** là bắt buộc với DP khó: chỉ cần một chỉ số lệch là sai mà ví dụ đề vẫn có thể đúng.

## 15. Biến thể

1. **Strange Printer (LeetCode 664):** in chuỗi bằng các lượt in đoạn cùng ký tự; cũng chọn ký tự đầu và hỏi "nó được in cùng ai" — nhưng chỉ cần `dp[l][r]`.
2. **Zuma Game (LeetCode 488):** chèn bóng, 3 bóng cùng màu thì nổ và hai phía khép lại — trạng thái phức tạp hơn, thường BFS/DFS có nhớ.
3. **Điểm là `k^3` hoặc `f(k)` lồi bất kỳ:** công thức giữ nguyên, chỉ thay `(k+1)^2` bằng `f(k+1)`. (Nếu `f` không lồi, bước gộp nhóm liền kề có thể không còn an toàn!)
4. **Chỉ được bỏ nhóm có ít nhất 2 hộp:** thêm điều kiện ở lựa chọn "bỏ ngay".
5. **Burst Balloons (LeetCode 312):** DP đoạn kinh điển với ý tưởng "phần tử bị xử lý **cuối cùng**".

In [29]:
# Biến thể 3: điểm = f(số hộp) với f bất kỳ lồi, ví dụ f(x) = x^3
def remove_boxes_f(boxes, f):
    n = len(boxes)
    @lru_cache(maxsize=None)
    def dp(l, r, k):
        if l > r:
            return 0
        while l < r and boxes[l + 1] == boxes[l]:
            l += 1; k += 1
        res = f(k + 1) + dp(l + 1, r, 0)
        for m in range(l + 1, r + 1):
            if boxes[m] == boxes[l]:
                res = max(res, dp(l + 1, m - 1, 0) + dp(m, r, k + 1))
        return res
    return dp(0, n - 1, 0)

print("x^2:", remove_boxes_f([1, 3, 2, 2, 2, 3, 4, 3, 1], lambda x: x * x))
print("x^3:", remove_boxes_f([1, 3, 2, 2, 2, 3, 4, 3, 1], lambda x: x ** 3))
assert remove_boxes_f([1, 3, 2, 2, 2, 3, 4, 3, 1], lambda x: x * x) == 23

x^2: 23
x^3: 63


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force tuple | Thử bỏ mọi đoạn cùng màu, nhớ theo dãy còn lại | `O(2^n · n^3)` | `O(2^n · n)` | Kiểm chứng, `n <= 12` |
| 2. `dp(l,r,k)` có nhớ | Hộp trái nhất + `k` hộp dính, bỏ ngay hoặc gộp với `m` | `O(n^4)` (thực tế ít) | `O(n^3)` | **Nộp bài**, dễ viết |
| 3. Bảng 3 chiều | Cùng công thức, điền `l` giảm dần | `Θ(n^4)` | `O(n^3)` | Hiểu thứ tự tính, ngôn ngữ nhanh (C++) |
| 4. Nhóm nén | DP trên nhóm `(màu, số lượng)` | `O(G^3 · n)` | `O(G^2 · n)` | Dãy có nhiều đoạn lặp, muốn nhanh hơn |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code, đoán `Solution().removeBoxes([3, 1, 3])` và `Solution().removeBoxes([1, 2, 3, 2, 1])`.

**Bài 2 (tính tay).** Tính tay `dp(0, 3, 0)` cho `boxes = [1, 2, 2, 1]` theo công thức (ghi rõ hai lựa chọn).

**Bài 3 (điền TODO).** Hoàn thiện hàm `count_states(boxes)` trả về số trạng thái **khác nhau** mà Cách 2 (có gộp) thực sự lưu vào cache.

**Bài 4 (tự viết lại).** Viết lại Cách 2 nhưng định nghĩa `k` là số hộp cùng màu `boxes[r]` dính **bên phải** đoạn (đối xứng). Hàm `remove_boxes_right(boxes)`.

**Bài 5 (mini challenge).** Ngoài điểm tối đa, hãy trả về **một chuỗi thao tác** đạt điểm đó: danh sách các nhóm bị bỏ theo thứ tự (mỗi nhóm là list màu). Hàm `remove_boxes_plan(boxes) -> (score, plan)`.

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def count_states(boxes):
    # TODO: viết dp(l, r, k) có lru_cache như Cách 2,
    #       gọi dp(0, n-1, 0), rồi trả về dp.cache_info().currsize
    pass

# Bài 4 — TODO
def remove_boxes_right(boxes):
    # TODO: dp(l, r, k) với k = số hộp cùng màu boxes[r] dính bên phải
    #       Lựa chọn: bỏ ngay (k+1)^2 + dp(l, r-1, 0)
    #       hoặc gộp với m < r cùng màu: dp(m+1, r-1, 0) + dp(l, m, k+1)
    pass

# Bài 5 — TODO
def remove_boxes_plan(boxes):
    # TODO: lưu lựa chọn tốt nhất của mỗi trạng thái, sau đó dựng lại kế hoạch
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: hộp ở giữa nên bỏ trước.
- Mức 2: `[3,1,3]` giống `[1,2,1]`.
- Mức 3: `[1,2,3,2,1]`: bỏ 3 (1), gộp 2,2 (4), gộp 1,1 (4).

**Bài 2.**
- Mức 1: hộp 0 màu 1, `k = 0`.
- Mức 2: bỏ ngay → `1 + dp(1,3,0)`; gộp với `m = 3` → `dp(1,2,0) + dp(3,3,1)`.
- Mức 3: `dp(1,3,0) = 4 + 1 = 5`; `dp(1,2,0) = 4`; `dp(3,3,1) = 4`.

**Bài 3.**
- Mức 1: chép Cách 2.
- Mức 2: `lru_cache` có `cache_info()`.
- Mức 3: `currsize` là số khóa đang lưu.

**Bài 4.**
- Mức 1: "chốt" là hộp **phải nhất**.
- Mức 2: gộp nhóm liền kề: `while l < r and boxes[r-1] == boxes[r]: r -= 1; k += 1`.
- Mức 3: đối chiếu với `Solution` trên dữ liệu ngẫu nhiên.

**Bài 5.**
- Mức 1: lưu `choice[(l,r,k)] = None` (bỏ ngay) hoặc `m`.
- Mức 2: hàm `build(l, r, k)` trả về danh sách nhóm theo thứ tự bỏ; khối ở `l` bị bỏ **sau** phần giữa.
- Mức 3: với lựa chọn `m`: kế hoạch = `build(l+1, m-1, 0)` + `build(m, r, k+1)` (khối của `l` nằm trong nhóm bỏ cuối của `m`, nên cần truyền thêm "các hộp dính"). Cách đơn giản: kiểm tra bằng `simulate` rằng tổng điểm đúng.

## 19. Đáp án tham khảo

**Bài 1:** `[3,1,3]` → 5; `[1,2,3,2,1]` → 9.

**Bài 2:** `dp(0,3,0) = max(1 + 5, 4 + 4) = 8`.

In [31]:
assert Solution().removeBoxes([3, 1, 3]) == 5
assert Solution().removeBoxes([1, 2, 3, 2, 1]) == 9
assert Solution().removeBoxes([1, 2, 2, 1]) == 8

# Bài 3
def count_states(boxes):
    n = len(boxes)
    @lru_cache(maxsize=None)
    def dp(l, r, k):
        if l > r:
            return 0
        while l < r and boxes[l + 1] == boxes[l]:
            l += 1; k += 1
        res = (k + 1) ** 2 + dp(l + 1, r, 0)
        for m in range(l + 1, r + 1):
            if boxes[m] == boxes[l]:
                res = max(res, dp(l + 1, m - 1, 0) + dp(m, r, k + 1))
        return res
    dp(0, n - 1, 0)
    return dp.cache_info().currsize

print("số trạng thái ví dụ 1:", count_states([1, 3, 2, 2, 2, 3, 4, 3, 1]))
assert count_states([1, 3, 2, 2, 2, 3, 4, 3, 1]) == remove_boxes_memo([1, 3, 2, 2, 2, 3, 4, 3, 1])[1]

số trạng thái ví dụ 1: 27


In [32]:
# Bài 4
def remove_boxes_right(boxes):
    n = len(boxes)
    @lru_cache(maxsize=None)
    def dp(l, r, k):
        if l > r:
            return 0
        while l < r and boxes[r - 1] == boxes[r]:
            r -= 1; k += 1
        res = (k + 1) ** 2 + dp(l, r - 1, 0)
        for m in range(l, r):
            if boxes[m] == boxes[r]:
                res = max(res, dp(m + 1, r - 1, 0) + dp(l, m, k + 1))
        return res
    return dp(0, n - 1, 0)

random.seed(4)
for _ in range(200):
    arr = [random.randint(1, 3) for _ in range(random.randint(1, 12))]
    assert remove_boxes_right(arr) == Solution().removeBoxes(arr), arr
print("Bài 4 OK")

Bài 4 OK


In [33]:
# Bài 5: dựng lại kế hoạch. Ta làm việc với VỊ TRÍ gốc để dễ kiểm tra.
def remove_boxes_plan(boxes):
    n = len(boxes)
    memo, choice = {}, {}

    def dp(l, r, k):
        if l > r:
            return 0
        key = (l, r, k)
        if key in memo:
            return memo[key]
        res, ch = (k + 1) ** 2 + dp(l + 1, r, 0), None
        for m in range(l + 1, r + 1):
            if boxes[m] == boxes[l]:
                v = dp(l + 1, m - 1, 0) + dp(m, r, k + 1)
                if v > res:
                    res, ch = v, m
        memo[key], choice[key] = res, ch
        return res

    def build(l, r, glued):
        # glued: danh sách VỊ TRÍ các hộp đang dính bên trái l. Trả về các lượt bỏ (list vị trí).
        if l > r:
            return []
        ch = choice[(l, r, len(glued))]
        if ch is None:
            return [glued + [l]] + build(l + 1, r, [])
        return build(l + 1, ch - 1, []) + build(ch, r, glued + [l])

    score = dp(0, n - 1, 0)
    plan_pos = build(0, n - 1, [])
    return score, plan_pos

def check_plan(boxes, plan_pos):
    """Thực hiện kế hoạch theo vị trí gốc, kiểm tra mỗi lượt là đoạn liền kề cùng màu."""
    alive = list(range(len(boxes)))
    total = 0
    for grp in plan_pos:
        idx = [alive.index(p) for p in grp]
        assert idx == list(range(idx[0], idx[0] + len(idx))), "không liền kề"
        assert len({boxes[p] for p in grp}) == 1, "khác màu"
        total += len(grp) ** 2
        alive = [p for p in alive if p not in grp]
    assert not alive
    return total

score, plan = remove_boxes_plan([1, 3, 2, 2, 2, 3, 4, 3, 1])
print(score, [[[1, 3, 2, 2, 2, 3, 4, 3, 1][p] for p in g] for g in plan])
assert check_plan([1, 3, 2, 2, 2, 3, 4, 3, 1], plan) == score == 23

random.seed(5)
for _ in range(200):
    arr = [random.randint(1, 3) for _ in range(random.randint(1, 10))]
    s, p = remove_boxes_plan(arr)
    assert s == Solution().removeBoxes(arr) == check_plan(arr, p), arr
print("Bài 5 OK")

23 [[2, 2, 2], [4], [3, 3, 3], [1, 1]]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "DP đoạn + số phần tử dính bên trái":**

```python
@lru_cache(None)
def dp(l, r, k):                      # k phần tử cùng loại a[l] đang dính bên trái
    if l > r: return 0
    while l < r and a[l+1] == a[l]:   # gộp phần tử liền kề cùng loại
        l += 1; k += 1
    best = f(k + 1) + dp(l + 1, r, 0) # xử lý ngay khối ở l
    for m in range(l + 1, r + 1):     # hoặc giữ lại để gộp với a[m]
        if a[m] == a[l]:
            best = max(best, dp(l + 1, m - 1, 0) + dp(m, r, k + 1))
    return best
```

**Câu thần chú:** *"Hộp trái nhất sẽ bị bỏ cùng ai? Dọn phần giữa, mang khối theo."*

**Checklist trước khi submit:**

- [ ] Trạng thái có đủ 3 chiều `(l, r, k)`? (`dp[l][r]` là sai.)
- [ ] Khi gộp nhóm liền kề: tăng **cả** `l` và `k`.
- [ ] Phần giữa gọi với `k = 0`; phần sau gọi với `k + 1`.
- [ ] Trường hợp `l > r` trả 0.
- [ ] `lru_cache` đặt trên hàm **lồng trong method** để không dính cache giữa các test.
- [ ] Đã thử `[1]`, toàn một màu, toàn khác màu, `[1,2,1]`.
- [ ] Đã đối chiếu với brute force trên dữ liệu ngẫu nhiên nhỏ.